#Reading Data From Bronze

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import DateType

In [0]:
df = spark.table('workspace.bronze.erp_px_cat')

In [0]:
display(df)

In [0]:
for field in df.schema.fields:
  print(field.name, field.dataType)

#Transforming Data

Transforming sales table
- trim string values - good practice
- normalization for status & gender
- column names are not friendly

##Trimming

In [0]:
#Triming the string column 

for field in df.schema.fields:
    if field.dataType == StringType():
        df_erp_cust = df.withColumn(field.name, trim(col(field.name)))

## Correcting the gender column

In [0]:
#standardizing the columns 
#lets get all the unique values to understand what all Values we have in gender column 
df.select("CAT").distinct().show()

In [0]:
df.select("SUBCAT").distinct().show()

In [0]:
df.select("MAINTENANCE").distinct().show()

In [0]:
#renaming the column names
remap_name = {
    "ID": "id",
    "CAT" : "category",
    "SUBCAT": "sub_category",
    "MAINTENANCE": "maintenance"
}
for old_val, new_val in remap_name.items():
    df = df.withColumnRenamed(old_val, new_val)
#validating
df.columns

#Writing to Silver 

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_cat")

In [0]:
%sql
select * from workspace.silver.erp_cat Limit 10